In [6]:
import gc
from pathlib import Path
import pickle
import pandas as pd

def fix_class_module(obj):
    """
    If `obj.__class__` references the old module, reassign it to a class that exists
    in your old environment (e.g. `pandas.core.indexes.base.Index`).
    """
    cls = getattr(obj, "__class__", None)
    if cls and cls.__module__ == "pandas.core.indexes.numeric":
        # If name is Float64Index or Int64Index, either map to an older class
        # or forcibly rename it to plain Index.
        if cls.__name__ in ("Float64Index", "Int64Index", "NumericIndex"):
            obj.__class__ = pd.Index  # or pd.Float64Index if available
        else:
            obj.__class__ = pd.Index  # fallback

def walk_and_fix(obj, visited=None):
    """
    Recursively walk through all container types (dict, list, DataFrame, etc.)
    and fix any sub-objects referencing the old module path.
    """
    if visited is None:
        visited = set()

    obj_id = id(obj)
    if obj_id in visited:
        return
    visited.add(obj_id)

    # Fix the object's own class if needed
    fix_class_module(obj)

    # If it's a DataFrame, fix its index, columns, etc.
    if isinstance(obj, pd.DataFrame):
        walk_and_fix(obj.index, visited)
        walk_and_fix(obj.columns, visited)
        # If it has a MultiIndex, you may need to walk that too, but usually just fix_class_module is enough

    # If it's a Series
    elif isinstance(obj, pd.Series):
        walk_and_fix(obj.index, visited)

    # If it's a dict, fix all values
    elif isinstance(obj, dict):
        for v in obj.values():
            walk_and_fix(v, visited)

    # If it's a list/tuple, fix all elements
    elif isinstance(obj, (list, tuple, set)):
        for v in obj:
            walk_and_fix(v, visited)

    # If there are other custom container classes, adapt as needed.
    # The main idea: any sub-object might hold a reference, so you must fix them all.

dirpath   = Path(r"D:\WORK\Salvador\repo\model_tuner\test_data\a1_ou_unconn\scott_2025_03_13")
fpath_in  = dirpath / "OUmapping_master.pkl"
fpath_out  = dirpath / "OUmapping_master_compat.pkl"

# Then do:
with open(fpath_in, "rb") as f:
    data = pickle.load(f)

walk_and_fix(data)

# Now re-check if you still have numeric references:
raw = pickle.dumps(data)
assert b"pandas.core.indexes.numeric" not in raw, "Still found numeric references!"

# If assertion passes, dump to file
with open(fpath_out, "wb") as f:
    f.write(raw)


In [2]:
from pathlib import Path
import pickle

dirpath   = Path(r"D:\WORK\Salvador\repo\model_tuner\test_data\a1_ou_unconn\scott_2025_03_13")
fpath_in  = dirpath / "OUmapping_master_compat.pkl"

with open(fpath_in, 'rb') as f:
    data = pickle.load(f)

